In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


# Data Cleansing

In [ ]:
# Making sure all columns show
import pandas as pd, numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE

from scipy.stats import pearsonr

df = pd.read_csv("adult.csv")

print(df.shape)
df.info()
df.describe(include='all')

In [ ]:
# For some reason this dataset has missing values as ?, so we're replacing them with NAN.
df = df.replace("?", np.nan)
# strip whitespace from all text columns
for c in df.select_dtypes(include=['object', 'str']).columns:
    df[c] = df[c].str.strip()

In [ ]:
# Drop all missing values
df = df.dropna()

In [ ]:
# Remove duplicates
df = df.drop_duplicates()

In [ ]:
# Dropping Education since education.num is essentially an encoded numeric representation that's easier to work with. Here is the matching:
'''
Preschool	1
1st-4th	2
5th-6th	3
7th-8th	4
9th	5
10th	6
11th	7
12th	8
HS-grad	9
Some-college	10
Assoc-voc	11
Assoc-acdm	12
Bachelors	13
Masters	14
Prof-school	15
Doctorate	16
'''
# Also dropping fnlwgt since its just a sampling weight. Additionally dropping capital gain/loss.
df = df.drop(columns=['education', 'fnlwgt', 'capital.gain', 'capital.loss'])

In [ ]:
# Encoding income to be binary, 0 if less than 50k, 1 if more.
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

In [ ]:
# Turning native country into either US or Other, since its 90% U.S
df['native.country'] = np.where(df['native.country']=='United-States', 'US', 'Other')

In [ ]:
# Turning all the string values into binary values (one hot encoding)

df_raw = df.copy() # making a copy for later

df = pd.get_dummies(df, columns=['workclass','marital.status','occupation',
                                 'relationship','race','sex','native.country'],
                    drop_first=True)


In [ ]:
print(*(f"{col}: {dtype}" for col, dtype in zip(df.columns, df.dtypes)), sep="\n")

In [ ]:
# Check for Outliers using boxplots
numerical_cols = df.select_dtypes(
    include=['int64', 'float64']
).columns.drop('income', errors='ignore')

for col in numerical_cols:
    plt.figure(figsize=(7, 3))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

In [ ]:
# Check for Outliers numerically
outlier_results = []

for col in numerical_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ]

    outlier_results.append({
        'Column': col,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound,
        'Outlier Count': len(outliers),
        'Outlier Percentage': round(
            len(outliers) / len(df) * 100, 2
        )
    })

outlier_summary = pd.DataFrame(outlier_results)

outlier_summary

In [ ]:
# Investigate outliers. Age and education.num outliers are realistic and not IMPOSSIBLE. Will check hours per week if anything exceeds 168
invalid_hours = df[df['hours.per.week'] > 168]
print("Invalid records:", len(invalid_hours))

In [ ]:
# Split dataset for training into 80/20 data,
from sklearn.model_selection import train_test_split
X = df.drop(columns=['income'])
y = df['income']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale numeric features and fit the scaler on the training data
from sklearn.preprocessing import StandardScaler
num_cols = ['age','education.num', 'hours.per.week']

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train, y_train
)

# Check class distribution
print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

In [ ]:
# Just making sure everything is good
print(X_train.isna().sum().sum(), "missing in train") 
print(X_train.dtypes.value_counts())
print(X_train.shape, X_test.shape)

# Analysis
Here we will analyze the data to gather relevant statistics.


In [ ]:
analysis_Y = df_raw["income"].astype(int) 
analysis_X = df_raw.drop(columns=["income"])

## Check for Class Imbalance

In [ ]:
print(analysis_Y.value_counts(normalize=True).round(3))

There appears to be some class imbalance, with 75% of the samples falling in the earnig less than 50k category.

### Check Pearson Correlation Coef against Target 
for Numerical features

In [ ]:
num_assoc = analysis_X[num_cols].corrwith(analysis_Y).sort_values(key=abs, ascending=False)
print(num_assoc.round(3))


No single feature has too high of a correlation against the target (>0.75), indicating that these features do not need to be removed for being overly correlated.

### Check Chi2 P value against Target

for Categorical features


In [ ]:
from scipy.stats import chi2_contingency

rows = []
for c in cat_cols:
    chi2, p, dof, _ = chi2_contingency(pd.crosstab(analysis_X[c], analysis_Y))
    rows.append({"column": c, "chi2": chi2, "dof": dof, "p_value": p})

chi_table = pd.DataFrame(rows).sort_values("chi2", ascending=False).reset_index(drop=True)
print(chi_table.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
num_assoc.sort_values().plot(kind="barh", ax=axes[0])

axes[0].set_title("Correlation with income")
chi_table.sort_values("chi2").plot(kind="barh", x="column", y="chi2", ax=axes[1], legend=False)

axes[1].set_title("Chi-square with income")
axes[1].set_xlabel("chi-square statistic")
plt.tight_layout()
plt.show()

According to the Chi-square statistics and p-values, every categorical variable shows signs of association with the target variable, meaning that we do not need to remove it for being irrelevant.

In [ ]:
for _, row in chi_table.iterrows():
    print(f"{row['column']}: is under 0.05? {row['p_value'] < 0.05}")

# Numeric columns (Pearson p-values)
for c in num_cols:
    r, p = pearsonr(analysis_X[c], analysis_Y)
    print(f"{c}: is under 0.05? {p < 0.05}")

In [ ]:
plot_df = analysis_X[num_cols].assign(income=analysis_Y)

for c in num_cols:
    plt.figure(figsize=(7, 4))
    sns.kdeplot(data=df_raw, x=c, hue="income", common_norm=False, fill=True)
    plt.title(f"{c} by income")
    plt.tight_layout()
    plt.show()

We plot the distributions of each numerical value for each population, to quickly glance and see if there are clear differences in distributions. This affirms that these are good features as the distributions between the two classes appear to be different, suggesting there is a correlation.

# Model 1: Ensemble Learning with Decision Trees

In [ ]:
# Exploring what the data actually looks like right now
df.describe(include='all')
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
print(df.shape[0])

In [ ]:
print(y_test[y_test== 1].count())

In [ ]:
# Looking at the distribution of classes
for col in df:
    print(df[col].value_counts(), "\n")

In [ ]:
clf = tree.DecisionTreeClassifier()
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap=plt.cm.Blues) 
plt.title("Confusion Matrix")
plt.show() 

In [ ]:
def predict_on_subset(col, val, model):
    df_subset = df[df[col] == val]
    X_subset = df_subset.drop(columns=['income'])
    y_subset = df_subset['income']
    X_train_subset, X_test_subset, y_train_subset, y_test_subset = train_test_split(
        X_subset, y_subset, test_size=0.2, random_state=42, stratify=y_subset)
    y_pred_subset = model.predict(X_test_subset)
    cm = confusion_matrix(y_test_subset, y_pred_subset)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(cmap=plt.cm.Blues) 
    print(classification_report(y_test, y_pred))
    plt.title("Confusion Matrix")
    plt.show() 


In [ ]:
predict_on_subset('race_Black', True, clf)

In [ ]:
predict_on_subset('race_Other', True, clf)

In [ ]:
def test_model(clf):
    y_pred = clf.predict(X_test)
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(cmap=plt.cm.Blues) 
    plt.title("Confusion Matrix")
    plt.show() 

In [ ]:
clf = tree.DecisionTreeClassifier(criterion='entropy')
clf.fit(X_train, y_train)
test_model(clf)
clf = tree.DecisionTreeClassifier(criterion='gini')
clf.fit(X_train, y_train)
test_model(clf)

In [ ]:
clf = tree.DecisionTreeClassifier(max_depth=5)
clf.fit(X_train, y_train)
test_model(clf)

In [ ]:
clf = tree.DecisionTreeClassifier(max_depth=15)
clf.fit(X_train, y_train)
test_model(clf)

In [ ]:
clf = tree.DecisionTreeClassifier(max_depth=10)
clf.fit(X_train, y_train)
test_model(clf)
clf = tree.DecisionTreeClassifier(max_depth=20)
clf.fit(X_train, y_train)
test_model(clf)

It looks like a decision tree with a maximum depth of 15 works the best with the highest f1 score and achieves the best fit for correctly identifying actual positive cases

In [ ]:
predict_on_subset('race_Black', True, clf)
predict_on_subset('race_Other', True, clf)

However, the limited depth is making it less prone to correctly predicting high income black earners which is problematic. I am going to try bagging next to see if it can provide a better fit

In [ ]:
from sklearn.ensemble import BaggingClassifier

In [ ]:
base_tree = tree.DecisionTreeClassifier(max_depth=15)
bagging_clf = bagging_clf = BaggingClassifier(estimator=base_tree, n_estimators=50, max_samples=0.8, bootstrap=True, random_state=42)
bagging_clf.fit(X_train, y_train)
test_model(bagging_clf)

In [ ]:
predict_on_subset('race_Black', True, bagging_clf)
predict_on_subset('race_Other', True, bagging_clf)

In [ ]:
base_tree = tree.DecisionTreeClassifier(max_depth=50)
bagging_clf = bagging_clf = BaggingClassifier(estimator=base_tree, n_estimators=50, max_samples=0.8, bootstrap=True, random_state=42)
bagging_clf.fit(X_train, y_train)
test_model(bagging_clf)
predict_on_subset('race_Black', True, bagging_clf)
predict_on_subset('race_Other', True, bagging_clf)

### Trying with class weights balanced

In [ ]:
clf = tree.DecisionTreeClassifier(max_depth=15, class_weight='balanced')
clf.fit(X_train, y_train)
test_model(clf)

In [ ]:
predict_on_subset('race_Black', True, clf)
predict_on_subset('race_Other', True, clf)

Balancing the class weights did a really good job at mitigating the racial bias within the model, but it did result in lower overall accuracy. I am going to try increasing the depth to see if that can help

In [ ]:
clf = tree.DecisionTreeClassifier(max_depth=20, class_weight='balanced')
clf.fit(X_train, y_train)
test_model(clf)
predict_on_subset('race_Black', True, clf)
predict_on_subset('race_Other', True, clf)

20 seems like a better depth now with a 0.87 F1 score compared to the previous 0.86 and only a minor decrease in accuracy for black earners. A depth of 15 still makes more sense if focusing on eliminating racial bias though.

In [ ]:
base_tree = tree.DecisionTreeClassifier(max_depth=15, class_weight='balanced')
bagging_clf = bagging_clf = BaggingClassifier(estimator=base_tree, n_estimators=50, max_samples=0.8, bootstrap=True, random_state=42)
bagging_clf.fit(X_train, y_train)
test_model(bagging_clf)
predict_on_subset('race_Black', True, bagging_clf)
predict_on_subset('race_Other', True, bagging_clf)

This led to a significant number of false positives

## Trying to change min_samples_split

In [ ]:
clf = tree.DecisionTreeClassifier(max_depth=15, class_weight='balanced')
clf.fit(X_train, y_train)
test_model(clf)
predict_on_subset('race_Black', True, clf)
predict_on_subset('race_Other', True, clf)

worsens false positives

## Attempting a Stacking Approach

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
estimators = [
    ('random_forest', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('decision_tree', DecisionTreeClassifier(max_depth=15, class_weight='balanced', random_state=42))
]
stacking_clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())
stacking_clf.fit(X_train, y_train)

In [ ]:
test_model(stacking_clf)
predict_on_subset('race_Black', True, stacking_clf)
predict_on_subset('race_Other', True, stacking_clf)

Using stacking led to a higher f1 score with more false negatives. Interestingly, false positives are now over represented in racial minorities while false negatives are significantly lower. False postives from black individuals make up almost 20% of the false positives even though black individuals only make up around 10% of the data's population

### Trying bagging the stacked algorithm, just because

In [ ]:
bagging_clf = BaggingClassifier(estimator=stacking_clf, n_estimators=50, max_samples=0.8, bootstrap=True, random_state=42)
bagging_clf.fit(X_train, y_train)
test_model(bagging_clf)
predict_on_subset('race_Black', True, bagging_clf)
predict_on_subset('race_Other', True, bagging_clf)

# Method 2: Logistic Regression with Grid Search



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score

# Define the grid search
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100], # regularization constant
    "l1_ratio": [0, 0.5, 1], # l1 ratio
    "class_weight": [None, "balanced"], # penalize according to class unbalance
}

grid = GridSearchCV(
    LogisticRegression(solver="saga", max_iter=1000),
    param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=2,
)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV F1:", round(grid.best_score_, 4))

y_pred = grid.best_estimator_.predict(X_test)
print("Test F1:", round(f1_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))


## Evaluate Logistic Regression model

In [ ]:
def evaluate_predictions(y_true, y_pred, title="Confusion Matrix"):
    print(classification_report(y_true, y_pred))
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["<=50K", ">50K"])
    disp.plot(cmap=plt.cm.Blues)
    plt.title(title)
    plt.show()


y_pred = grid.best_estimator_.predict(X_test)
evaluate_predictions(y_test, y_pred, "Logistic Regression")

# Method 3: Support Vector Machines

In [ ]:
svm_clf = SVC()
svm_clf.fit(X_train, y_train)
test_model(svm_clf)

The default SVM (rbf kernel, C=1) gets 86% accuracy, but it only catches 62% of the people earning more than 50k (recall of 0.62 for class 1). I am going to try other kernels to see if that helps

## Trying different kernels

In [ ]:
svm_clf = SVC(kernel='linear')
svm_clf.fit(X_train, y_train)
test_model(svm_clf)
svm_clf = SVC(kernel='poly')
svm_clf.fit(X_train, y_train)
test_model(svm_clf)

Neither kernel helped. Linear and polynomial both did slightly worse on the >50k class (F1 of 0.68 compared to 0.69), so I am sticking with rbf

### Trying with class weights balanced

In [ ]:
svm_clf = SVC(class_weight='balanced')
svm_clf.fit(X_train, y_train)
test_model(svm_clf)

Balancing the class weights raised the recall for >50k from 0.62 to 0.87, so the model misses far fewer high earners (198 false negatives compared to 575). The tradeoff is a lot more false positives (964 compared to 266) and lower overall accuracy (81% compared to 86%). F1 stayed about the same at 0.69

## Trying different values of C

In [ ]:
svm_clf = SVC(C=0.1, class_weight='balanced')
svm_clf.fit(X_train, y_train)
test_model(svm_clf)
svm_clf = SVC(C=10, class_weight='balanced')
svm_clf.fit(X_train, y_train)
test_model(svm_clf)

Changing C barely made a difference. C=0.1 dropped accuracy to 80%, and C=10 performed about the same as C=1 while taking longer to train

## Trying different values of gamma

In [ ]:
svm_clf = SVC(gamma=0.01, class_weight='balanced')
svm_clf.fit(X_train, y_train)
test_model(svm_clf)
svm_clf = SVC(gamma=0.1, class_weight='balanced')
svm_clf.fit(X_train, y_train)
test_model(svm_clf)

gamma=0.01 was slightly worse. gamma=0.1 gave the exact same results as the default, which makes sense since the default gamma ('scale') works out to roughly 0.1 for this data

## Using Grid Search
To double check the settings I picked by hand, I am running a grid search over the kernel, C and class weights

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score

# Define the grid search
param_grid = {
    "C": [0.1, 1, 10], # regularization constant
    "kernel": ["linear", "rbf"], # kernel type
    "class_weight": [None, "balanced"], # penalize according to class unbalance
}

svm_grid = GridSearchCV(
    SVC(),
    param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=2,
)
svm_grid.fit(X_train, y_train)

print("Best params:", svm_grid.best_params_)
print("Best CV F1:", round(svm_grid.best_score_, 4))

y_pred = svm_grid.best_estimator_.predict(X_test)
print("Test F1:", round(f1_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))


## Evaluate SVM model

In [ ]:
y_pred = svm_grid.best_estimator_.predict(X_test)
evaluate_predictions(y_test, y_pred, "SVM")

The grid search agreed with what I found by hand: the rbf kernel with C=1 and balanced class weights gave the best F1 (0.69 on the test set). It catches 87% of high earners, but only 57% of the people it flags as earning more than 50k actually do